# 02 — Data Preprocessing
Clean, scale, and engineer features. Save processed datasets to `data/processed/`.

In [ ]:
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
import pandas as pd
import matplotlib.pyplot as plt
import config
from src.preprocess import load_data, handle_missing_values, remove_outliers, scale_features
from src.features import add_behavioral_features
%matplotlib inline

## 1. Load Raw Data

In [ ]:
X_train, y_train = load_data(config.X_TRAIN_PATH, config.Y_TRAIN_PATH)
X_test = pd.read_csv(config.X_TEST_PATH)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

## 2. Handle Missing Values

In [ ]:
print('Missing before:', X_train.isnull().sum().sum())
X_train = handle_missing_values(X_train)
X_test  = handle_missing_values(X_test)
print('Missing after :', X_train.isnull().sum().sum())

## 3. Remove Outliers

In [ ]:
X_train = remove_outliers(X_train, threshold=10.0)
X_test  = remove_outliers(X_test,  threshold=10.0)
print('Outliers clipped. Shape:', X_train.shape)

## 4. Scale Features

In [ ]:
X_train_scaled, X_test_scaled = scale_features(X_train, X_test)
print('Train mean (should ~0):', round(X_train_scaled.mean().mean(), 4))
print('Train std  (should ~1):', round(X_train_scaled.std().mean(), 4))

## 5. Feature Engineering

In [ ]:
X_train_eng = add_behavioral_features(X_train_scaled)
X_test_eng  = add_behavioral_features(X_test_scaled)
new_cols = [c for c in X_train_eng.columns if c not in X_train_scaled.columns]
print(f'Added {len(new_cols)} engineered features:', new_cols)
X_train_eng[new_cols].describe()

## 6. Save Processed Data

In [ ]:
os.makedirs(config.DATA_PROCESSED_DIR, exist_ok=True)
X_train_eng.to_csv(config.X_TRAIN_PROCESSED, index=False)
X_test_eng.to_csv(config.X_TEST_PROCESSED,   index=False)
print('Saved to', config.DATA_PROCESSED_DIR)

## 7. Before vs After Scaling

In [ ]:
cols = ['amount', 'session_duration', 'pin_entry_speed']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
X_train[cols].plot(kind='hist', bins=40, alpha=0.6, ax=axes[0], title='Before Scaling')
X_train_scaled[cols].plot(kind='hist', bins=40, alpha=0.6, ax=axes[1], title='After Scaling')
plt.tight_layout()
plt.show()